# 02.3 - PyTorch CNN voi gene_group split

Notebook nay giu cung input/model voi notebook 02, nhung split theo `GeneSymbol` de khong cho cung gene xuat hien o ca train/val/test. Muc tieu la kiem tra CNN co tong quat sang gene chua thay trong train hay khong.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"

X_PATH = PROCESSED_DIR / "X_ref_alt_marker_201.npy"
Y_PATH = PROCESSED_DIR / "y.npy"
METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"

for path in [X_PATH, Y_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Chua co file: {path}. Hay chay notebook 01 den buoc 12B sau khi them tensor ref/alt/marker."
        )

X_PATH, Y_PATH, METADATA_PATH

(PosixPath('/mnt/MyData/Bioinformatics/Project/processed/X_ref_alt_marker_201.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/y.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet'))

## 1. Load dataset da preprocess

`X` co shape `(n_variants, 201, 9)`.

9 channels gom:

- `0:4`: one-hot `ref_seq` theo thu tu A/C/G/T
- `4:8`: one-hot `alt_seq` theo thu tu A/C/G/T
- `8`: mutation marker, bang 1 tai vi tri center index 100

PyTorch `Conv1d` can input `(batch, channels, length)`, nen trong `Dataset` se transpose thanh `(9, 201)`.

In [2]:
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata_df = pd.read_parquet(METADATA_PATH)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"metadata shape: {metadata_df.shape}")
print(f"X dtype: {X.dtype}")
print(f"y dtype: {y.dtype}")

assert X.ndim == 3
assert X.shape[1:] == (201, 9)
assert len(X) == len(y) == len(metadata_df)

pd.Series(y).value_counts().rename(index={0: "Benign/Likely benign", 1: "Pathogenic/Likely pathogenic"})

X shape: (460488, 201, 9)
y shape: (460488,)
metadata shape: (460488, 15)
X dtype: uint8
y dtype: int8


Benign/Likely benign            401054
Pathogenic/Likely pathogenic     59434
Name: count, dtype: int64

## 2. Tao train/validation/test split theo gene

Khac notebook 02 random theo row, notebook nay dung `GroupShuffleSplit` theo `GeneSymbol`. Cac gene `unknown` neu co se duoc tach thanh group rieng theo row de khong gom mot group qua lon.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_STATE = 42
SPLIT_MODE = "gene_group"

# De None de train full. Doi thanh so nho, vi du 20_000, neu chi muon smoke test pipeline.
DEBUG_MAX_TRAIN_ROWS = None
DEBUG_MAX_VAL_ROWS = None
DEBUG_MAX_TEST_ROWS = None

indices = np.arange(len(y))
groups = metadata_df["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
unknown_mask = groups == "unknown"
groups[unknown_mask] = np.array([f"unknown_{i}" for i in np.where(unknown_mask)[0]])

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(indices, y, groups=groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
rel_val_idx, rel_test_idx = next(gss2.split(temp_idx, y[temp_idx], groups=groups[temp_idx]))
val_idx = temp_idx[rel_val_idx]
test_idx = temp_idx[rel_test_idx]


def maybe_subsample(idx, max_rows, name):
    if max_rows is None or len(idx) <= max_rows:
        return idx
    sampled, _ = train_test_split(
        idx,
        train_size=max_rows,
        random_state=RANDOM_STATE,
        stratify=y[idx],
    )
    print(f"DEBUG: {name} subset {len(idx):,} -> {len(sampled):,}")
    return sampled

train_idx = maybe_subsample(train_idx, DEBUG_MAX_TRAIN_ROWS, "train")
val_idx = maybe_subsample(val_idx, DEBUG_MAX_VAL_ROWS, "val")
test_idx = maybe_subsample(test_idx, DEBUG_MAX_TEST_ROWS, "test")

print("SPLIT_MODE:", SPLIT_MODE)
print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

train_genes = set(metadata_df.iloc[train_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
val_genes = set(metadata_df.iloc[val_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
test_genes = set(metadata_df.iloc[test_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
print("gene overlap train/val:", len(train_genes & val_genes))
print("gene overlap train/test:", len(train_genes & test_genes))
print("gene overlap val/test:", len(val_genes & test_genes))

key_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
def variant_keys(idx):
    return set(map(tuple, metadata_df.iloc[idx][key_cols].astype(str).to_numpy()))
print("variant key overlap train/val:", len(variant_keys(train_idx) & variant_keys(val_idx)))
print("variant key overlap train/test:", len(variant_keys(train_idx) & variant_keys(test_idx)))


SPLIT_MODE: gene_group
train: 319,608, labels={np.int8(0): np.int64(278577), np.int8(1): np.int64(41031)}
val:   72,308, labels={np.int8(0): np.int64(62401), np.int8(1): np.int64(9907)}
test:  68,572, labels={np.int8(0): np.int64(60076), np.int8(1): np.int64(8496)}
gene overlap train/val: 0
gene overlap train/test: 0
gene overlap val/test: 0
variant key overlap train/val: 0
variant key overlap train/test: 0


## 2B. Audit khoang cach genome train/test

Gene split khong dam bao genomic distance xa tuyet doi, nhung se loai bo shortcut cung gene. Cell nay do nhanh nearest train distance tren cung chromosome.


In [4]:
train_pos_by_chr = {}
for chrom, sub in metadata_df.iloc[train_idx].groupby("Chromosome", sort=False):
    train_pos_by_chr[str(chrom)] = np.sort(
        pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna().astype(np.int64).to_numpy()
    )


def nearest_train_distances(idx):
    chunks = []
    for chrom, sub in metadata_df.iloc[idx].groupby("Chromosome", sort=False):
        arr = train_pos_by_chr.get(str(chrom))
        if arr is None or len(arr) == 0:
            continue
        pos = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna().astype(np.int64).to_numpy()
        j = np.searchsorted(arr, pos)
        d = np.full(len(pos), np.iinfo(np.int64).max, dtype=np.int64)
        m = j < len(arr)
        d[m] = np.minimum(d[m], np.abs(arr[j[m]] - pos[m]))
        m = j > 0
        d[m] = np.minimum(d[m], np.abs(arr[j[m] - 1] - pos[m]))
        chunks.append(d)
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.int64)

for split_name, idx in [("val", val_idx), ("test", test_idx)]:
    d = nearest_train_distances(idx)
    print("\n", split_name)
    print("quantiles:", {q: int(np.quantile(d, q)) for q in [0, .01, .05, .10, .25, .50, .75, .90, .95, .99]})
    for t in [1, 10, 50, 100, 201, 500, 1000, 10000]:
        print(f"pct <= {t:5,d} bp: {(d <= t).mean():.4f}")



 val
quantiles: {0: 0, 0.01: 121, 0.05: 2271, 0.1: 6396, 0.25: 16440, 0.5: 36009, 0.75: 76095, 0.9: 168264, 0.95: 216252, 0.99: 567150}
pct <=     1 bp: 0.0005
pct <=    10 bp: 0.0022
pct <=    50 bp: 0.0067
pct <=   100 bp: 0.0091
pct <=   201 bp: 0.0122
pct <=   500 bp: 0.0190
pct <= 1,000 bp: 0.0265
pct <= 10,000 bp: 0.1505

 test
quantiles: {0: 0, 0.01: 209, 0.05: 3649, 0.1: 6751, 0.25: 17656, 0.5: 38932, 0.75: 112705, 0.9: 241188, 0.95: 405967, 0.99: 849232}
pct <=     1 bp: 0.0005
pct <=    10 bp: 0.0017
pct <=    50 bp: 0.0048
pct <=   100 bp: 0.0069
pct <=   201 bp: 0.0098
pct <=   500 bp: 0.0150
pct <= 1,000 bp: 0.0199
pct <= 10,000 bp: 0.1513


## 3. Tao PyTorch Dataset/DataLoader

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Imbalance handling.
USE_WEIGHTED_SAMPLER = True
POS_WEIGHT_MODE = "none"  # "none", "sqrt", hoac "full"; none vi USE_WEIGHTED_SAMPLER=True de tranh double class-weight

# Hard-negative fine-tuning.
USE_HARD_NEGATIVE_FINETUNE = True
HARD_NEGATIVE_THRESHOLD = 0.50
HARD_NEGATIVE_EPOCHS = 2
HARD_NEGATIVE_LR = 3e-4
HARD_NEGATIVE_MAX_EASY_NEG_RATIO = 1.0


class ClinVarSequenceDataset(Dataset):
    def __init__(self, X_array, y_array, indices):
        self.X_array = X_array
        self.y_array = y_array
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = self.indices[item]
        # X raw: (201, 9). Conv1d needs (channels, length) = (9, 201).
        # np.load(..., mmap_mode="r") returns read-only arrays, so copy before torch.from_numpy.
        x_np = np.asarray(self.X_array[idx]).T.copy()
        x = torch.from_numpy(x_np).float()
        target = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, target


def make_weighted_sampler(indices):
    labels = y[np.asarray(indices)]
    class_count = np.bincount(labels, minlength=2)
    class_weight = 1.0 / np.maximum(class_count, 1)
    sample_weight = class_weight[labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weight),
        num_samples=len(indices),
        replacement=True,
    )


BATCH_SIZE = 512

train_dataset = ClinVarSequenceDataset(X, y, train_idx)
train_sampler = make_weighted_sampler(train_idx) if USE_WEIGHTED_SAMPLER else None

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ClinVarSequenceDataset(X, y, val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ClinVarSequenceDataset(X, y, test_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)
print("USE_WEIGHTED_SAMPLER:", USE_WEIGHTED_SAMPLER)
print("POS_WEIGHT_MODE:", POS_WEIGHT_MODE)
print("USE_HARD_NEGATIVE_FINETUNE:", USE_HARD_NEGATIVE_FINETUNE)


## 4. Build CNN 1D PyTorch

Kien truc demo:

- Conv1d + BatchNorm + ReLU + MaxPool
- Conv1d + BatchNorm + ReLU + MaxPool
- Conv1d + BatchNorm + ReLU
- AdaptiveMaxPool
- Dense output logits

In [6]:
class ClinVarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(9, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)


model = ClinVarCNN().to(DEVICE)
model


ClinVarCNN(
  (features): Sequential(
    (0): Conv1d(9, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveMaxPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=Fa

## 5. Train nhanh

Dung `BCEWithLogitsLoss`. Neu class imbalance, `pos_weight` se tang/giam trong loss cho class `1`.

In [ ]:
# PyTorch/sympy compatibility guard.
# In some long-running notebook kernels, sympy.core is not attached to the sympy module
# when torch.optim lazily imports torch._dynamo. Force-load it before creating optimizer.
import importlib
import sympy

if not hasattr(sympy, "core"):
    sympy.core = importlib.import_module("sympy.core")
if not hasattr(sympy.core, "symbol"):
    sympy.core.symbol = importlib.import_module("sympy.core.symbol")

positive_count = (y_train := y[train_idx]).sum()
negative_count = len(y_train) - positive_count
full_pos_weight = negative_count / max(positive_count, 1)

if POS_WEIGHT_MODE == "none":
    pos_weight_value = 1.0
elif POS_WEIGHT_MODE == "sqrt":
    pos_weight_value = float(np.sqrt(full_pos_weight))
elif POS_WEIGHT_MODE == "full":
    pos_weight_value = float(full_pos_weight)
else:
    raise ValueError(f"POS_WEIGHT_MODE khong hop le: {POS_WEIGHT_MODE}")

if pos_weight_value == 1.0:
    pos_weight = torch.tensor([1.0], dtype=torch.float32, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss()
else:
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"full_pos_weight: {full_pos_weight:.4f}")
print(f"used_pos_weight: {pos_weight.item():.4f}")


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score


def run_epoch(model, loader, train=False, epoch_label=""):
    model.train(train)
    total_loss = 0.0
    all_targets = []
    all_probs = []
    mode = "train" if train else "eval"

    progress = tqdm(loader, desc=f"{mode} {epoch_label}".strip(), unit="batch", leave=False)
    for batch_x, batch_y in progress:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(batch_y.detach().cpu().numpy())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    targets = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(np.int8)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": (preds == targets).mean(),
        "roc_auc": roc_auc_score(targets, probs),
        "pr_auc": average_precision_score(targets, probs),
    }
    return metrics, probs


EPOCHS = 5
best_val_auc = -np.inf
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_metrics, _ = run_epoch(model, train_loader, train=True, epoch_label=f"main {epoch}/{EPOCHS}")
    val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"main {epoch}/{EPOCHS}")

    row = {"phase": "main", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

    if val_metrics["pr_auc"] > best_val_auc:
        best_val_auc = val_metrics["pr_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"New best val_pr_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

if USE_HARD_NEGATIVE_FINETUNE:
    print()
    print("Hard-negative mining on train set...")
    _, train_probs_ordered = run_epoch(model, train_eval_loader, train=False, epoch_label="mine train")
    train_labels_ordered = y[train_idx]
    hard_neg_mask = (train_labels_ordered == 0) & (train_probs_ordered >= HARD_NEGATIVE_THRESHOLD)
    pos_idx = train_idx[train_labels_ordered == 1]
    hard_neg_idx = train_idx[hard_neg_mask]
    easy_neg_idx = train_idx[(train_labels_ordered == 0) & ~hard_neg_mask]

    rng = np.random.default_rng(RANDOM_STATE)
    max_easy = int((len(pos_idx) + len(hard_neg_idx)) * HARD_NEGATIVE_MAX_EASY_NEG_RATIO)
    if len(easy_neg_idx) > max_easy:
        easy_neg_idx = rng.choice(easy_neg_idx, size=max_easy, replace=False)

    hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
    rng.shuffle(hard_train_idx)

    print(f"positive train rows: {len(pos_idx):,}")
    print(f"hard negative rows:  {len(hard_neg_idx):,}")
    print(f"easy negative rows:  {len(easy_neg_idx):,}")
    print(f"fine-tune rows:      {len(hard_train_idx):,}")

    hard_loader = DataLoader(
        ClinVarSequenceDataset(X, y, hard_train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=HARD_NEGATIVE_LR, weight_decay=1e-4)
    for epoch in range(1, HARD_NEGATIVE_EPOCHS + 1):
        print(f"Hard-negative epoch {epoch}/{HARD_NEGATIVE_EPOCHS}")
        train_metrics, _ = run_epoch(model, hard_loader, train=True, epoch_label=f"hard {epoch}/{HARD_NEGATIVE_EPOCHS}")
        val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"hard {epoch}/{HARD_NEGATIVE_EPOCHS}")
        row = {"phase": "hard_negative", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

        if val_metrics["pr_auc"] > best_val_auc:
            best_val_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"New best val_pr_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df


Epoch 1/5


train main 1/5:   0%|          | 0/625 [00:00<?, ?batch/s]

eval main 1/5:   0%|          | 0/142 [00:00<?, ?batch/s]

{'phase': 'main', 'epoch': 1, 'train_loss': 0.9072, 'train_accuracy': np.float64(0.6178), 'train_roc_auc': 0.753, 'train_pr_auc': 0.7495, 'val_loss': 0.8283, 'val_accuracy': np.float64(0.5987), 'val_roc_auc': 0.8075, 'val_pr_auc': 0.447}
New best val_pr_auc: 0.4470
Epoch 2/5


train main 2/5:   0%|          | 0/625 [00:00<?, ?batch/s]

...

{'phase': 'hard_negative', 'epoch': 1, 'train_loss': 0.5279, 'train_accuracy': np.float64(0.8129), 'train_roc_auc': 0.8796, 'train_pr_auc': 0.6081, 'val_loss': 0.5929, 'val_accuracy': np.float64(0.8522), 'val_roc_auc': 0.8243, 'val_pr_auc': 0.498}
New best val_pr_auc: 0.4980
Hard-negative epoch 2/2


train hard 2/2:   0%|          | 0/457 [00:00<?, ?batch/s]

eval hard 2/2:   0%|          | 0/142 [00:00<?, ?batch/s]

{'phase': 'hard_negative', 'epoch': 2, 'train_loss': 0.5049, 'train_accuracy': np.float64(0.8221), 'train_roc_auc': 0.8901, 'train_pr_auc': 0.6328, 'val_loss': 0.6073, 'val_accuracy': np.float64(0.8619), 'val_roc_auc': 0.8232, 'val_pr_auc': 0.4977}


,phase,epoch,train_loss,train_accuracy,train_roc_auc,train_pr_auc,val_loss,val_accuracy,val_roc_auc,val_pr_auc
0,main,1,0.907205,0.617847,0.752968,0.749490,0.828324,0.598661,0.807493,0.446964
1,main,2,0.749163,0.729853,0.847842,0.837070,0.616694,0.777853,0.817313,0.468309
2,main,3,0.683880,0.761317,0.872707,0.858259,0.702509,0.694861,0.825205,0.485013
3,main,4,0.636266,0.783125,0.888613,0.872845,0.661940,0.731205,0.825039,0.493138
4,main,5,0.593953,0.799780,0.900875,0.884782,0.687236,0.729145,0.823479,0.494693
5,hard_negative,1,0.527929,0.812868,0.879563,0.608139,0.592940,0.852160,0.824322,0.498031
6,hard_negative,2,0.504927,0.822076,0.890097,0.632795,0.607270,0.861869,0.823216,0.497690


## 6. Danh gia tren test set

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

test_metrics, proba_test = run_epoch(model, test_loader, train=False, epoch_label="test")
pred_test = (proba_test >= 0.5).astype(np.int8)
y_test = y[test_idx]

print(test_metrics)
print("Classification report:")
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"], zero_division=0))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df


eval test:   0%|          | 0/134 [00:00<?, ?batch/s]

{'loss': 0.5268346254858831, 'accuracy': np.float64(0.8593740885492621), 'roc_auc': 0.8397266681365562, 'pr_auc': 0.5113702942041977}
Classification report:
              precision    recall  f1-score   support

      benign       0.94      0.90      0.92     60076
  pathogenic       0.45      0.56      0.50      8496

    accuracy                           0.86     68572
   macro avg       0.69      0.73      0.71     68572
weighted avg       0.88      0.86      0.87     68572



,pred_benign,pred_pathogenic
true_benign,54148,5928
true_pathogenic,3715,4781


## 6A. Threshold table cho precision/recall

Thay vi dung co dinh threshold `0.5`, cell nay tinh bang threshold de chon operating point: uu tien recall pathogenic hay precision pathogenic.


In [10]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
rows = []
for threshold, p_value, r_value in zip(thresholds, precision[:-1], recall[:-1]):
    f1 = 2 * p_value * r_value / (p_value + r_value) if (p_value + r_value) else 0.0
    beta = 2
    f2 = (1 + beta**2) * p_value * r_value / (beta**2 * p_value + r_value) if (beta**2 * p_value + r_value) else 0.0
    rows.append({
        "threshold": float(threshold),
        "precision": float(p_value),
        "recall": float(r_value),
        "f1": float(f1),
        "f2": float(f2),
    })

threshold_df = pd.DataFrame(rows)

for target_precision in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    candidates = threshold_df[threshold_df["precision"] >= target_precision]
    best_recall = candidates["recall"].max() if len(candidates) else 0.0
    print(f"Recall at precision >= {target_precision:.2f}: {best_recall:.4f}")

for target_recall in [0.50, 0.60, 0.70, 0.80, 0.90]:
    candidates = threshold_df[threshold_df["recall"] >= target_recall]
    best_precision = candidates["precision"].max() if len(candidates) else 0.0
    print(f"Precision at recall >= {target_recall:.2f}: {best_precision:.4f}")

print("Best F1 threshold:")
display(threshold_df.sort_values("f1", ascending=False).head(1))
print("Best F2 threshold:")
display(threshold_df.sort_values("f2", ascending=False).head(1))

display(threshold_df.iloc[np.linspace(0, len(threshold_df) - 1, 12, dtype=int)])


Recall at precision >= 0.30: 0.7920
Recall at precision >= 0.40: 0.6335
Recall at precision >= 0.50: 0.4828
Recall at precision >= 0.60: 0.3500
Recall at precision >= 0.70: 0.2446
Recall at precision >= 0.80: 0.1328
Precision at recall >= 0.50: 0.4883
Precision at recall >= 0.60: 0.4243
Precision at recall >= 0.70: 0.3574
Precision at recall >= 0.80: 0.2937
Precision at recall >= 0.90: 0.2128
Best F1 threshold:


,threshold,precision,recall,f1,f2
57077,0.481851,0.435796,0.585217,0.499573,0.547662


Best F2 threshold:


,threshold,precision,recall,f1,f2
46898,0.260533,0.307806,0.782486,0.441816,0.598035


,threshold,precision,recall,f1,f2
0,0.000027,0.123899,1.000000,0.220481,0.414213
6224,0.003821,0.135139,0.991643,0.237863,0.437312
12449,0.008634,0.148301,0.979284,0.257593,0.461781
18674,0.016283,0.163605,0.960099,0.279570,0.486451
24899,0.029155,0.182760,0.938324,0.305933,0.513633
31124,0.053239,0.206275,0.907721,0.336159,0.540275
37349,0.101709,0.236824,0.868409,0.372156,0.566336
43574,0.191866,0.278544,0.817208,0.415475,0.589289
49799,0.321997,0.334332,0.735640,0.459728,0.593227
56024,0.454020,0.416707,0.611229,0.495563,0.559036


## 6B. Cach doc ket qua gene_group split

Ket qua o notebook nay se thuong thap hon notebook 02 random split. Do la dieu binh thuong va dang mong doi, vi test set gom cac gene khong xuat hien trong train.

Nen so sanh cac chi so sau voi notebook 02:

- `PR-AUC`: quan trong nhat cho class pathogenic thua so.
- `ROC-AUC`: do kha nang xep hang tong quat.
- pathogenic precision/recall trong classification report.
- confusion matrix, dac biet false negative pathogenic.

Neu gene_group split tut manh, ket qua random split truoc do dang optimistic do cung gene/cung neighborhood.


## 7. Xem mot vai prediction kem metadata

In [11]:
test_metadata_df = metadata_df.iloc[test_idx].copy()
test_metadata_df["pred_proba_pathogenic"] = proba_test
test_metadata_df["pred_label"] = pred_test

display(
    test_metadata_df[[
        "GeneSymbol",
        "ClinicalSignificance",
        "Y",
        "pred_proba_pathogenic",
        "pred_label",
        "Chromosome",
        "PositionVCF",
        "ReferenceAlleleVCF",
        "AlternateAlleleVCF",
        "ReviewStatus",
    ]]
    .sort_values("pred_proba_pathogenic", ascending=False)
    .head(20)
)

,GeneSymbol,ClinicalSignificance,Y,pred_proba_pathogenic,pred_label,Chromosome,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,ReviewStatus
268389,NDUFAF5,Likely pathogenic,1,0.997076,1,20,13787353,G,C,"criteria provided, single submitter"
281051,PDE6C,Likely pathogenic,1,0.996283,1,10,93655861,G,C,"criteria provided, single submitter"
180444,AGRN,Likely benign,0,0.995587,1,1,1051237,C,A,"criteria provided, single submitter"
57791,HNF1B,Pathogenic,1,0.995212,1,17,37710502,C,G,"criteria provided, single submitter"
301195,AGL,Likely pathogenic,1,0.995079,1,1,99888109,G,C,"criteria provided, single submitter"
177933,RB1,Pathogenic,1,0.994434,1,13,48349024,G,C,"criteria provided, multiple submitters, no con..."
426289,RERE,Likely pathogenic,1,0.994420,1,1,8541213,C,A,"criteria provided, single submitter"
379904,CTNS,Likely pathogenic,1,0.994308,1,17,3648932,G,C,"criteria provided, single submitter"
433459,TRIM63,Likely pathogenic,1,0.994169,1,1,26067335,C,T,"criteria provided, single submitter"
2108,RB1,Pathogenic,1,0.994014,1,13,48349024,G,T,"criteria provided, multiple submitters, no con..."


## 8. Luu model, predictions, history va threshold table


In [ ]:
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "clinvar_ref_alt_marker_cnn_gene_group_balanced_pytorch.pt"
prediction_path = PROCESSED_DIR / "cnn_gene_group_balanced_test_predictions_pytorch.parquet"
history_path = PROCESSED_DIR / "cnn_gene_group_balanced_training_history_pytorch.parquet"
threshold_path = PROCESSED_DIR / "cnn_gene_group_balanced_threshold_table_pytorch.parquet"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_class": "ClinVarCNN",
        "input_shape": (9, 201),
        "channels": ["ref_A", "ref_C", "ref_G", "ref_T", "alt_A", "alt_C", "alt_G", "alt_T", "mutation_marker"],
        "threshold": 0.5,
        "split_mode": SPLIT_MODE,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "pos_weight_mode": POS_WEIGHT_MODE,
        "use_hard_negative_finetune": USE_HARD_NEGATIVE_FINETUNE,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "test_metrics": test_metrics,
    },
    model_path,
)

test_metadata_df.to_parquet(prediction_path, index=False, engine="pyarrow")
history_df.to_parquet(history_path, index=False, engine="pyarrow")
threshold_df.to_parquet(threshold_path, index=False, engine="pyarrow")

print(f"Saved model: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved history: {history_path}")
print(f"Saved threshold table: {threshold_path}")
